# Model 2 - Face Recognition

Pipeline theo ESP-WHO: **face crop -> FaceNet embedding 512D -> cosine similarity với gallery đã đăng ký**.

Notebook này fine-tune FaceNet pretrained VGGFace2 bằng softmax classifier đơn giản. Classifier chỉ phục vụ train; khi điểm danh chỉ dùng embedding 512D.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 160
MAX_IDENTITIES = 3000
MAX_IMAGES_PER_IDENTITY = 60
MIN_IMAGES_PER_IDENTITY = 12
ENROLL_IMAGES = 5
EPOCHS = 40
BATCH_SIZE = 128
WORKERS = 6
LEARNING_RATE = 2e-5

INPUT_ROOT = Path('/kaggle/input/datasets')
WEBFACE_ROOT = INPUT_ROOT / 'yakhyokhuja' / 'webface-112x112' / 'webface_112x112'
FACENET_ROOT = INPUT_ROOT / 'timesler' / 'facenet-pytorch-vggface2'
OUTPUT_DIR = Path('/kaggle/working/model-2-recognition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_IDS = list(range(torch.cuda.device_count()))
assert WEBFACE_ROOT.is_dir(), WEBFACE_ROOT
assert FACENET_ROOT.is_dir(), FACENET_ROOT
print({'device': str(DEVICE), 'gpus': [torch.cuda.get_device_name(index) for index in GPU_IDS], 'epochs': EPOCHS, 'image_size': IMAGE_SIZE})

In [ ]:
# EDA: WebFace gồm các thư mục identity; một thư mục là một người.
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
identity_dirs = sorted(path for path in WEBFACE_ROOT.iterdir() if path.is_dir())
image_counts = {path.name: len([image for image in path.iterdir() if image.suffix.lower() in IMAGE_EXTENSIONS]) for path in identity_dirs}
usable_ids = [name for name, count in image_counts.items() if count >= MIN_IMAGES_PER_IDENTITY]
random.Random(SEED).shuffle(usable_ids)
usable_ids = usable_ids[:MAX_IDENTITIES]
assert len(usable_ids) == MAX_IDENTITIES, 'WebFace dataset has too few usable identities'
print({'all_identities': len(identity_dirs), 'selected_identities': len(usable_ids), 'min_images': min(image_counts.values()), 'max_images': max(image_counts.values())})

sample_paths = [next(path for path in (WEBFACE_ROOT / identity).iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS) for identity in usable_ids[:6]]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for axis, path in zip(axes, sample_paths):
    axis.imshow(Image.open(path))
    axis.set_title(path.parent.name)
    axis.axis('off')
plt.show()

In [ ]:
# Preprocess: WebFace đã align. Khi deploy, Model 1 YOLO tạo face crop tương đương trước cell này.
import subprocess
wheel = next(FACENET_ROOT.glob('facenet_pytorch-*.whl'))
subprocess.run(['pip', '-q', 'install', str(wheel)], check=True)
from facenet_pytorch import fixed_image_standardization

rows = []
for label, identity in enumerate(usable_ids):
    images = [path for path in sorted((WEBFACE_ROOT / identity).iterdir()) if path.suffix.lower() in IMAGE_EXTENSIONS][:MAX_IMAGES_PER_IDENTITY]
    random.Random(f'{SEED}-{identity}').shuffle(images)
    val_count = max(1, round(len(images) * 0.2))
    rows.extend({'path': path, 'label': label, 'identity': identity, 'split': 'val' if index < val_count else 'train'} for index, path in enumerate(images))

train_rows = [row for row in rows if row['split'] == 'train']
val_rows = [row for row in rows if row['split'] == 'val']
print({'train_images': len(train_rows), 'val_images': len(val_rows), 'classes': len(usable_ids)})

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    fixed_image_standardization,
])
eval_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), fixed_image_standardization])

class FaceDataset(Dataset):
    def __init__(self, records, transform):
        self.records, self.transform = records, transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(row['path']) as image:
            return self.transform(image.convert('RGB')), row['label']

loader_args = {'batch_size': BATCH_SIZE, 'num_workers': WORKERS, 'pin_memory': DEVICE.type == 'cuda', 'persistent_workers': True}
train_loader = DataLoader(FaceDataset(train_rows, train_transform), shuffle=True, **loader_args)
val_loader = DataLoader(FaceDataset(val_rows, eval_transform), shuffle=False, **loader_args)

In [ ]:
# Train: pretrained FaceNet + ArcFace head. The head is train-only; deployment keeps 512-D embeddings.
from facenet_pytorch import InceptionResnetV1

class ArcFaceHead(nn.Module):
    def __init__(self, classes, scale=32.0, margin=0.35):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, 512))
        nn.init.xavier_uniform_(self.weight)
        self.scale, self.margin = scale, margin
    def forward(self, embeddings, labels=None):
        cosine = torch.nn.functional.linear(torch.nn.functional.normalize(embeddings), torch.nn.functional.normalize(self.weight)).clamp(-1 + 1e-7, 1 - 1e-7)
        if labels is None:
            return cosine * self.scale
        target = torch.cos(torch.acos(cosine) + self.margin)
        mask = torch.nn.functional.one_hot(labels, num_classes=cosine.size(1)).float()
        return (cosine * (1 - mask) + target * mask) * self.scale

weights_path = FACENET_ROOT / '20180402-114759-vggface2-features.pth'
embedder = InceptionResnetV1(classify=False, pretrained=None, num_classes=8631)
embedder.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=True), strict=False)
classifier = ArcFaceHead(len(usable_ids))
embedder, classifier = embedder.to(DEVICE), classifier.to(DEVICE)
if len(GPU_IDS) > 1:
    embedder = nn.DataParallel(embedder, device_ids=GPU_IDS)
    classifier = nn.DataParallel(classifier, device_ids=GPU_IDS)
for parameter in embedder.parameters():
    parameter.requires_grad = False
optimizer = torch.optim.AdamW([{'params': embedder.parameters(), 'lr': LEARNING_RATE}, {'params': classifier.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
history, best_accuracy = [], 0.0

for epoch in range(1, EPOCHS + 1):
    if epoch == 4:
        for parameter in embedder.parameters():
            parameter.requires_grad = True
    embedder.train(); classifier.train()
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(classifier(embedder(images), labels), labels)
        loss.backward(); optimizer.step()

    embedder.eval(); classifier.eval(); correct = total = 0
    with torch.inference_mode():
        for images, labels in val_loader:
            predictions = classifier(embedder(images.to(DEVICE))).argmax(1).cpu()
            correct += (predictions == labels).sum().item(); total += labels.numel()
    accuracy = correct / total
    history.append({'epoch': epoch, 'val_accuracy': accuracy})
    print(history[-1]); scheduler.step()
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        state_dict = embedder.module.state_dict() if isinstance(embedder, nn.DataParallel) else embedder.state_dict()
        torch.save({'state_dict': state_dict, 'image_size': IMAGE_SIZE, 'embedding_dim': 512}, OUTPUT_DIR / 'facenet_best.pt')

In [ ]:
# Output: gallery demo. Production thay ảnh WebFace bằng 5 ảnh enrollment của từng sinh viên.
checkpoint = torch.load(OUTPUT_DIR / 'facenet_best.pt', map_location=DEVICE, weights_only=True)
embedder = InceptionResnetV1(classify=False, pretrained=None, num_classes=8631).to(DEVICE)
embedder.load_state_dict(checkpoint['state_dict'], strict=False)
embedder.eval()

@torch.inference_mode()
def embed(path):
    with Image.open(path) as image:
        pixels = eval_transform(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    return torch.nn.functional.normalize(embedder(pixels), dim=1)[0].cpu().numpy()

gallery = {}
for identity in usable_ids:
    enrolled = [row['path'] for row in train_rows if row['identity'] == identity][:ENROLL_IMAGES]
    vector = np.mean([embed(path) for path in enrolled], axis=0)
    gallery[identity] = vector / np.linalg.norm(vector)

names, gallery_matrix = list(gallery), np.stack(list(gallery.values()))
correct = total = 0
for row in val_rows:
    predicted = names[int((gallery_matrix @ embed(row['path'])).argmax())]
    correct += predicted == row['identity']; total += 1

np.save(OUTPUT_DIR / 'gallery_demo_512d.npy', gallery_matrix)
(OUTPUT_DIR / 'gallery_demo_labels.json').write_text(json.dumps(names, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps({'classifier_val_accuracy': best_accuracy, 'gallery_top1_accuracy': correct / total, 'history': history}, indent=2), encoding='utf-8')
print({'classifier_val_accuracy': best_accuracy, 'gallery_top1_accuracy': correct / total, 'artifacts': [path.name for path in OUTPUT_DIR.iterdir()]})